In [166]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [167]:
train_df = pd.read_csv("train.csv")
public_df = pd.read_csv("public_test.csv")

print("Training Set Shape:", train_df.shape)
print("Public Test Shape:", public_df.shape)

train_df.head()

Training Set Shape: (240, 5)
Public Test Shape: (400, 5)


,id,text,label,label_name,source_file
0,pos_cv230_7428,"well , i'll admit when i first heard about thi...",1,positive,pos/cv230_7428.txt
1,pos_cv853_29233,my summer was recently saved by two very diffe...,1,positive,pos/cv853_29233.txt
2,pos_cv771_28665,in october of 1962 the united states found its...,1,positive,pos/cv771_28665.txt
3,pos_cv449_8785,this is a good year if you want plenty of sci-...,1,positive,pos/cv449_8785.txt
4,pos_cv130_17083,"while watching wes anderson's rushmore , it ma...",1,positive,pos/cv130_17083.txt


In [168]:
print(train_df["label"].value_counts())

label
1    180
0     60
Name: count, dtype: int64


In [169]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))

Training samples: 192
Validation samples: 48


Dataset ~

The training dataset has a total of 240 movie reviews. I started by splitting the data into 80% for training which is 192 reviews and 20% for validation making it 48 reviews. I used a stratified split so the positive and negative reviews stayed in the same proportion as the original dataset. This gives the model a fair way to evaluate its performance during training.

Model Structure ~

I used DistilBERT, which is a pretrained transformer model for natural language processing. The movie reviews are first converted into tokens that the model can understand. The tokenized reviews are then passed through DistilBERT, and the classification layer predicts whether the review is negative (0) or positive (1).

In [170]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [171]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

public_encodings = tokenizer(
    public_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenization complete!")

Tokenization complete!


In [172]:
from torch.utils.data import Dataset

class SentimentDataset(Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels.iloc[idx]))

        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [173]:
train_labels = train_labels.reset_index(drop=True)
val_labels = val_labels.reset_index(drop=True)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)
public_dataset = SentimentDataset(public_encodings)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Public test dataset:", len(public_dataset))

Training dataset: 192
Validation dataset: 48
Public test dataset: 400


Handling the Small and Imbalanced Training Set

The training dataset is small and contains more positive reviews than negative reviews. To help prevent the model from favoring the positive class, I used class weights during training. This gives more importance to mistakes made on the smaller negative class. I also used a validation split so I could check the model during training and reduce the chance of overfitting.

In [174]:
class_counts = train_labels.value_counts().sort_index()

total_samples = len(train_labels)
num_classes = len(class_counts)

class_weights = total_samples / (num_classes * class_counts)

print("Class counts:")
print(class_counts)

print("\nClass weights:")
print(class_weights)

Class counts:
label
0     48
1    144
Name: count, dtype: int64

Class weights:
label
0    2.000000
1    0.666667
Name: count, dtype: float64


Training Settings

I used the AdamW optimizer with a learning rate of 2e-5 because it is commonly used for fine-tuning transformer models. I trained the model with a batch size of 8 for 3 epochs. The model was evaluated on the validation dataset after training.

In [175]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3588.53it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [176]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

weights = torch.tensor(
    class_weights.values,
    dtype=torch.float
).to(device)

loss_function = nn.CrossEntropyLoss(weight=weights)

batch_size = 8
learning_rate = 2e-5
epochs = 3

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size
)

optimizer = AdamW(
    model.parameters(),
    lr=learning_rate
)

print("Device:", device)
print("Batch size:", batch_size)
print("Learning rate:", learning_rate)
print("Epochs:", epochs)
print("Class weights:", weights)

Device: cpu
Batch size: 8
Learning rate: 2e-05
Epochs: 3
Class weights: tensor([2.0000, 0.6667])


In [177]:
# Train the model approach
for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Training Loss: {average_loss:.4f}")

KeyboardInterrupt: 

In [ ]:
model.eval()

val_predictions = []
val_true_labels = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_true_labels.extend(labels.cpu().numpy())

validation_accuracy = accuracy_score(
    val_true_labels,
    val_predictions
)

print(f"Validation Accuracy: {validation_accuracy:.4f}")

Validation Accuracy: 0.7500


In [ ]:
# Evaluate on the public test set

public_loader = DataLoader(
    public_dataset,
    batch_size=batch_size
)

model.eval()

public_predictions = []
public_true_labels = public_df["label"].tolist()

with torch.no_grad():

    for batch in public_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        public_predictions.extend(
            predictions.cpu().numpy()
        )

public_accuracy = accuracy_score(
    public_true_labels,
    public_predictions
)

print(f"Public Test Accuracy: {public_accuracy:.4f}")

Public Test Accuracy: 0.5250


In [178]:
public_cm = confusion_matrix(
    public_true_labels,
    public_predictions
)

print("Public Test Confusion Matrix:")
print(public_cm)

Public Test Confusion Matrix:
[[ 12 188]
 [  2 198]]


In [179]:
# Save the trained model and tokenizer
checkpoint_path = "model_checkpoint"

model.save_pretrained(checkpoint_path)
tokenizer.save_pretrained(checkpoint_path)

print("Model checkpoint saved!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]

Model checkpoint saved!


In [180]:
# Save public test predictions
predictions_df = pd.DataFrame({
    "id": public_df["id"],
    "predicted_label": public_predictions
})

predictions_df.to_csv(
    "public_test_predictions.csv",
    index=False
)

predictions_df.head()

,id,predicted_label
0,pos_cv696_29740,1
1,pos_cv669_22995,1
2,neg_cv963_7208,1
3,pos_cv182_7281,1
4,pos_cv162_10424,1


In [181]:
print(predictions_df.shape)
print(predictions_df["predicted_label"].value_counts())

(400, 2)
predicted_label
1    386
0     14
Name: count, dtype: int64


Evaluation Results

The model reached 75% accuracy on the validation set and 52.5% accuracy on the public test set. The confusion matrix showed that the model predicted most reviews as positive. It correctly classified most positive reviews, but it had difficulty identifying negative reviews. This is most likely because the training set had many more positive reviews than negative reviews and only contained 240 total reviews. Even though I used class weights to reduce the imbalance, the model still showed a bias toward the positive class.